# Notebook 08: Sensitivity and Robustness Analyses

## Purpose

Evaluate threshold sensitivity, complete-case selection, descriptive weighting, model-class consistency, and the consequences of removing race/ethnicity from the predictor set while retaining it for subgroup auditing.

## Inputs

- Completed outputs and fitted models from Notebooks 01--07

## Outputs

- Threshold-sensitivity, complete-case, weighting, and model-class summaries
- No-race model predictions, subgroup comparisons, and race-omission figure
- `data/processed/sensitivity_analysis_metadata.json`

## Dependencies

Run Notebooks 01--07 first. Notebook 09 loads the aggregate sensitivity outputs produced here.

> **Repository policy:** Notebook outputs and execution counts are cleared in the public source files. Run the notebooks in the documented order to regenerate all results.

## 1. Setup

In [ ]:
from pathlib import Path
import hashlib
import inspect
import json
import math
import os
import sys
import time
import warnings

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    import interpret
    from interpret.glassbox import ExplainableBoostingClassifier
except ImportError as exc:
    raise ImportError(
        "Notebook 08 requires the same InterpretML installation as Notebook 05. "
        "Install `interpret-core==0.7.8`, restart the kernel, and run again."
    ) from exc

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 220)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("InterpretML:", interpret.__version__)
print("Matplotlib:", matplotlib.__version__)


## 2. Configuration

The required no-race analysis is enabled by default.

For a quick technical test, temporarily set the environment variable `NOTEBOOK08_BOOTSTRAP_REPLICATES` below 1,000. Final saved results should use 1,000 replicates.

In [ ]:
RANDOM_STATE = 26
N_SPLITS = 5
INNER_THRESHOLD_SPLITS = 3
TARGET_SENSITIVITY_LEVELS = [0.70, 0.80, 0.90]
PRIMARY_TARGET_SENSITIVITY = 0.80
N_BOOTSTRAP = int(os.environ.get("NOTEBOOK08_BOOTSTRAP_REPLICATES", "1000"))

RUN_RACE_OMISSION_ANALYSIS = True

TARGET_COLUMNS = [
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
]
TARGET_DISPLAY_NAMES = {
    "self_reported_prior_diagnosis": "Prior reported clinician diagnosis",
    "current_hba1c_ge_6_5": "Current HbA1c at least 6.5%",
}
MODEL_NAMES = ["logistic", "ebm"]
MODEL_DISPLAY_NAMES = {
    "logistic": "Logistic regression",
    "ebm": "Explainable Boosting Machine",
}

CONTINUOUS_PREDICTORS = ["age", "bmi", "income_poverty_ratio"]
PRIMARY_CATEGORICAL_PREDICTORS = ["sex", "race_ethnicity", "insurance_history"]
NO_RACE_CATEGORICAL_PREDICTORS = ["sex", "insurance_history"]
PRIMARY_PREDICTORS = CONTINUOUS_PREDICTORS + PRIMARY_CATEGORICAL_PREDICTORS
NO_RACE_PREDICTORS = CONTINUOUS_PREDICTORS + NO_RACE_CATEGORICAL_PREDICTORS

SEX_LEVELS = ["Female", "Male"]
RACE_ETHNICITY_LEVELS = [
    "Mexican American",
    "Other Hispanic",
    "Non-Hispanic White",
    "Non-Hispanic Black",
    "Non-Hispanic Asian",
    "Other or multiracial",
]
INSURANCE_HISTORY_LEVELS = [
    "Continuously insured",
    "Currently insured, past-year gap",
    "Currently uninsured",
]
INCOME_GROUP_LEVELS = ["Below 1", "1 to below 2", "2 to below 4", "4 or higher"]

REFERENCE_CATEGORIES = {
    "sex": "Female",
    "race_ethnicity": "Non-Hispanic White",
    "income_poverty_group": "4 or higher",
    "insurance_history": "Continuously insured",
}
SUBGROUP_LEVELS = {
    "sex": SEX_LEVELS,
    "race_ethnicity": RACE_ETHNICITY_LEVELS,
    "income_poverty_group": INCOME_GROUP_LEVELS,
    "insurance_history": INSURANCE_HISTORY_LEVELS,
}
THRESHOLD_METRICS = [
    "false_negative_rate",
    "false_positive_rate",
    "positive_prediction_rate",
]

print("Bootstrap replicates:", N_BOOTSTRAP)
print("Required race-omission analysis:", RUN_RACE_OMISSION_ANALYSIS)


## 3. Project paths and upstream files

The notebook uses the same project structure as Notebooks 01–07. It first checks the expected relative path and then searches for the same basename while excluding copied final outputs.

In [ ]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
TABLE_DIR = PROJECT_DIR / "outputs" / "tables"
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
MODEL_DIR = PROJECT_DIR / "outputs" / "models"
FINAL_DIR = PROJECT_DIR / "outputs" / "final"
FINAL_MAIN_DIR = FINAL_DIR / "main"
FINAL_APPENDIX_DIR = FINAL_DIR / "appendix"

for directory in [
    PROCESSED_DIR, TABLE_DIR, FIGURE_DIR, MODEL_DIR,
    FINAL_MAIN_DIR, FINAL_APPENDIX_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

def resolve_project_file(relative_path: str, required: bool = True):
    expected = PROJECT_DIR / relative_path
    if expected.exists():
        return expected

    basename = Path(relative_path).name
    matches = [
        path for path in PROJECT_DIR.rglob(basename)
        if "outputs/final" not in path.as_posix()
    ]
    preferred = [
        path for path in matches
        if path.as_posix().endswith(relative_path.replace("\\", "/"))
    ]
    if len(preferred) == 1:
        return preferred[0]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple files named {basename!r} were found:\n"
            + "\n".join(f"- {path}" for path in matches)
        )
    if required:
        raise FileNotFoundError(
            f"Required file not found: {expected}\n"
            "Run the upstream notebook that creates it before Notebook 08."
        )
    return None

def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

LABELLED_DATA_PATH = resolve_project_file(
    "data/processed/nhanes_diabetes_complete_case_labeled.csv"
)
ANALYSIS_BASE_PATH = resolve_project_file(
    "data/processed/nhanes_diabetes_analysis_base.csv"
)
FOLD_ASSIGNMENT_PATH = resolve_project_file(
    "data/processed/primary_cv_fold_assignments.csv"
)
LOGISTIC_OOF_PATH = resolve_project_file(
    "data/processed/logistic_oof_predictions.csv"
)
EBM_OOF_PATH = resolve_project_file(
    "data/processed/ebm_oof_predictions.csv"
)
THRESHOLDED_LONG_PATH = resolve_project_file(
    "data/processed/thresholded_oof_predictions_long.csv"
)
SAMPLE_METADATA_PATH = resolve_project_file("data/processed/sample_metadata.json")
DESCRIPTIVE_METADATA_PATH = resolve_project_file(
    "data/processed/descriptive_analysis_metadata.json"
)
LOGISTIC_METADATA_PATH = resolve_project_file(
    "data/processed/logistic_regression_metadata.json"
)
EBM_METADATA_PATH = resolve_project_file(
    "data/processed/ebm_analysis_metadata.json"
)
PERFORMANCE_METADATA_PATH = resolve_project_file(
    "data/processed/performance_and_thresholds_metadata.json"
)
FAIRNESS_METADATA_PATH = resolve_project_file(
    "data/processed/fairness_and_subgroup_metadata.json",
    required=False,
)

print("Project directory:", PROJECT_DIR)
print("Analytic data:", LABELLED_DATA_PATH)
print("Fixed folds:", FOLD_ASSIGNMENT_PATH)


## 4. Load and validate the completed primary analysis

Notebook 08 does not recreate the analytic sample or the primary folds. All participant-level inputs are aligned by ID and checked before sensitivity models are fitted.

In [ ]:
data = pd.read_csv(LABELLED_DATA_PATH)
analysis_base = pd.read_csv(ANALYSIS_BASE_PATH)
fold_assignments = pd.read_csv(FOLD_ASSIGNMENT_PATH)
logistic_oof_raw = pd.read_csv(LOGISTIC_OOF_PATH)
ebm_oof_raw = pd.read_csv(EBM_OOF_PATH)
primary_thresholded_long = pd.read_csv(THRESHOLDED_LONG_PATH)

sample_metadata = load_json(SAMPLE_METADATA_PATH)
descriptive_metadata = load_json(DESCRIPTIVE_METADATA_PATH)
logistic_metadata = load_json(LOGISTIC_METADATA_PATH)
ebm_metadata = load_json(EBM_METADATA_PATH)
performance_metadata = load_json(PERFORMANCE_METADATA_PATH)
fairness_metadata = (
    load_json(FAIRNESS_METADATA_PATH)
    if FAIRNESS_METADATA_PATH is not None
    else {}
)

required_columns = {
    "id", "joint_label_code", "age", "bmi", "income_poverty_ratio",
    "sex", "race_ethnicity", "insurance_history", "phlebotomy_weight",
    *TARGET_COLUMNS,
}
missing = required_columns.difference(data.columns)
if missing:
    raise ValueError(f"Analytic data is missing columns: {sorted(missing)}")
if data["id"].duplicated().any() or fold_assignments["id"].duplicated().any():
    raise ValueError("Participant IDs must be unique.")
if set(data["id"]) != set(fold_assignments["id"]):
    raise ValueError("Analytic IDs and saved fold-assignment IDs differ.")

if "cv_fold" in data.columns:
    check = data[["id", "cv_fold"]].merge(
        fold_assignments[["id", "cv_fold"]],
        on="id",
        suffixes=("_data", "_file"),
        validate="one_to_one",
    )
    if not (
        check["cv_fold_data"].astype(int)
        == check["cv_fold_file"].astype(int)
    ).all():
        raise ValueError("Embedded folds differ from the saved primary folds.")
else:
    data = data.merge(
        fold_assignments[["id", "cv_fold"]],
        on="id",
        how="left",
        validate="one_to_one",
    )

data["cv_fold"] = data["cv_fold"].astype(int)
if sorted(data["cv_fold"].unique()) != list(range(1, N_SPLITS + 1)):
    raise ValueError("The expected five fixed folds are not present.")

data["income_poverty_group"] = pd.cut(
    data["income_poverty_ratio"],
    bins=[-np.inf, 1, 2, 4, np.inf],
    labels=INCOME_GROUP_LEVELS,
    right=False,
).astype(str)

for column, levels in {
    "sex": SEX_LEVELS,
    "race_ethnicity": RACE_ETHNICITY_LEVELS,
    "insurance_history": INSURANCE_HISTORY_LEVELS,
    "income_poverty_group": INCOME_GROUP_LEVELS,
}.items():
    unexpected = set(data[column].dropna().astype(str)).difference(levels)
    if unexpected:
        raise ValueError(f"Unexpected levels in {column}: {sorted(unexpected)}")

for target in TARGET_COLUMNS:
    if set(data[target].astype(int).unique()) != {0, 1}:
        raise ValueError(f"{target} is not a complete binary target.")

if data[PRIMARY_PREDICTORS + TARGET_COLUMNS].isna().any().any():
    raise ValueError("Primary predictors or targets contain missing values.")
if len(data) != int(descriptive_metadata.get("complete_case_n", len(data))):
    raise ValueError("Sample size differs from Notebook 03 metadata.")

print("Adult analysis base:", len(analysis_base))
print("Primary analytic sample:", len(data))
print("Fixed folds:", sorted(data["cv_fold"].unique()))
print(data["joint_label_code"].value_counts().sort_index())


## 5. Shared metric, weighting, threshold, and alignment helpers

In [ ]:
def safe_divide(numerator, denominator):
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    result = np.full(np.broadcast(numerator, denominator).shape, np.nan)
    np.divide(numerator, denominator, out=result, where=denominator != 0)
    return float(result) if result.ndim == 0 else result

def percentile_interval(values):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return np.nan, np.nan
    return tuple(np.percentile(finite, [2.5, 97.5]).astype(float))

def calibration_intercept_slope(outcome, probability, max_iter=100, tolerance=1e-10):
    y = np.asarray(outcome, dtype=float)
    p = np.clip(np.asarray(probability, dtype=float), 1e-8, 1 - 1e-8)
    if np.unique(y).size < 2:
        return np.nan, np.nan
    z = np.log(p / (1 - p))
    design = np.column_stack([np.ones(len(y)), z])
    prevalence = np.clip(y.mean(), 1e-8, 1 - 1e-8)
    beta = np.array([math.log(prevalence / (1 - prevalence)), 1.0])
    for _ in range(max_iter):
        linear = design @ beta
        mean = 1 / (1 + np.exp(-np.clip(linear, -35, 35)))
        weights = np.clip(mean * (1 - mean), 1e-10, None)
        information = design.T @ (design * weights[:, None])
        score = design.T @ (y - mean)
        try:
            step = np.linalg.solve(information, score)
        except np.linalg.LinAlgError:
            return np.nan, np.nan
        beta_new = beta + step
        if np.max(np.abs(beta_new - beta)) < tolerance:
            beta = beta_new
            break
        beta = beta_new
    return float(beta[0]), float(beta[1])

def probability_metrics(outcome, probability):
    y = np.asarray(outcome, dtype=int)
    p = np.asarray(probability, dtype=float)
    if np.isnan(p).any() or ((p < 0) | (p > 1)).any():
        raise ValueError("Probabilities must be complete and lie in [0, 1].")
    prevalence = float(y.mean())
    brier = float(brier_score_loss(y, p))
    null_brier = prevalence * (1 - prevalence)
    intercept, slope = calibration_intercept_slope(y, p)
    return {
        "prevalence": prevalence,
        "roc_auc": float(roc_auc_score(y, p)),
        "pr_auc": float(average_precision_score(y, p)),
        "brier_score": brier,
        "scaled_brier_score": 1 - brier / null_brier if null_brier > 0 else np.nan,
        "calibration_intercept": intercept,
        "calibration_slope": slope,
    }

def threshold_metrics(outcome, predicted_class):
    y = np.asarray(outcome, dtype=int)
    pred = np.asarray(predicted_class, dtype=int)
    tp = int(((y == 1) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum())
    fp = int(((y == 0) & (pred == 1)).sum())
    tn = int(((y == 0) & (pred == 0)).sum())
    return {
        "positive_n": int((y == 1).sum()),
        "negative_n": int((y == 0).sum()),
        "predicted_positive_n": int((pred == 1).sum()),
        "false_negative_rate": safe_divide(fn, tp + fn),
        "false_positive_rate": safe_divide(fp, fp + tn),
        "sensitivity": safe_divide(tp, tp + fn),
        "specificity": safe_divide(tn, tn + fp),
        "positive_prediction_rate": float(pred.mean()),
    }

def highest_threshold_reaching_sensitivity(outcome, probability, target_sensitivity):
    y = np.asarray(outcome, dtype=int)
    p = np.asarray(probability, dtype=float)
    positive_n = int((y == 1).sum())
    if positive_n == 0:
        raise ValueError("Threshold selection requires positive outcomes.")
    order = np.argsort(-p, kind="mergesort")
    cumulative = np.cumsum(y[order] == 1) / positive_n
    reached = np.flatnonzero(cumulative >= target_sensitivity)
    if reached.size == 0:
        return float(np.nextafter(p.min(), -np.inf))
    return float(p[order][int(reached[0])])

def weighted_mean(values, weights):
    values = pd.to_numeric(values, errors="coerce").to_numpy(float)
    weights = pd.to_numeric(weights, errors="coerce").to_numpy(float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    return float(np.average(values[valid], weights=weights[valid])) if valid.any() else np.nan

def weighted_sd(values, weights):
    values = pd.to_numeric(values, errors="coerce").to_numpy(float)
    weights = pd.to_numeric(weights, errors="coerce").to_numpy(float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan
    mean = np.average(values[valid], weights=weights[valid])
    return float(np.sqrt(np.average((values[valid] - mean) ** 2, weights=weights[valid])))

def weighted_quantile(values, weights, quantile):
    values = pd.to_numeric(values, errors="coerce").to_numpy(float)
    weights = pd.to_numeric(weights, errors="coerce").to_numpy(float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan
    values, weights = values[valid], weights[valid]
    order = np.argsort(values)
    values, weights = values[order], weights[order]
    cumulative = np.cumsum(weights)
    position = min(np.searchsorted(cumulative, quantile * cumulative[-1]), len(values) - 1)
    return float(values[position])

def infer_probability_column(frame, target, metadata):
    mapping = metadata.get("oof_probability_columns", {})
    if isinstance(mapping, dict):
        candidate = mapping.get(target)
        if isinstance(candidate, str) and candidate in frame.columns:
            return candidate
    candidates = [
        f"oof_probability_{target}",
        f"{target}_oof_probability",
        f"probability_{target}",
        f"{target}_probability",
    ]
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate
    semantic = [
        column for column in frame.columns
        if target in column
        and ("probability" in column.lower() or "prediction" in column.lower())
        and "class" not in column.lower()
    ]
    if len(semantic) == 1:
        return semantic[0]
    raise KeyError(f"Could not identify OOF probability column for {target}.")

def align_oof(frame, metadata, model_name):
    if frame["id"].duplicated().any():
        raise ValueError(f"{model_name} OOF predictions contain duplicate IDs.")
    aligned = data[["id", "cv_fold", *TARGET_COLUMNS]].copy()
    for target in TARGET_COLUMNS:
        column = infer_probability_column(frame, target, metadata)
        aligned = aligned.merge(
            frame[["id", column]].rename(
                columns={column: f"{model_name}__{target}__probability"}
            ),
            on="id",
            how="left",
            validate="one_to_one",
        )
    return aligned

primary_logistic_probabilities = align_oof(logistic_oof_raw, logistic_metadata, "logistic")
primary_ebm_probabilities = align_oof(ebm_oof_raw, ebm_metadata, "ebm")
primary_probabilities = primary_logistic_probabilities.merge(
    primary_ebm_probabilities[
        ["id", *[f"ebm__{target}__probability" for target in TARGET_COLUMNS]]
    ],
    on="id",
    how="left",
    validate="one_to_one",
)
probability_columns = [
    f"{model}__{target}__probability"
    for model in MODEL_NAMES for target in TARGET_COLUMNS
]
if primary_probabilities[probability_columns].isna().any().any():
    raise ValueError("At least one primary OOF probability is missing.")
if not (
    (primary_probabilities[probability_columns] >= 0)
    & (primary_probabilities[probability_columns] <= 1)
).all().all():
    raise ValueError("At least one primary OOF probability lies outside [0, 1].")

print("Aligned primary OOF probabilities:", primary_probabilities.shape)


## Section 12A.1 — Threshold sensitivity

The 80% training-sensitivity operating point is primary. This section compares subgroup false-negative rates, false-positive rates, positive prediction rates, and reference-group gaps at 70%, 80%, and 90%.

The stability labels are descriptive aids:

- **stable:** range no larger than 5 percentage points;
- **moderately threshold-sensitive:** range above 5 and no larger than 10 percentage points;
- **threshold-dependent:** range above 10 percentage points or a gap changes sign.

In [ ]:
required_threshold_columns = {
    "id", "cv_fold", "model", "target", "target_training_sensitivity",
    "probability", "selected_threshold", "predicted_class", "observed_outcome",
}
missing = required_threshold_columns.difference(primary_thresholded_long.columns)
if missing:
    raise ValueError(f"Thresholded file is missing columns: {sorted(missing)}")

primary_thresholded_long = primary_thresholded_long.copy()
if primary_thresholded_long["target_training_sensitivity"].max() > 1:
    primary_thresholded_long["target_training_sensitivity"] /= 100

expected_rows = (
    len(data) * len(MODEL_NAMES) * len(TARGET_COLUMNS)
    * len(TARGET_SENSITIVITY_LEVELS)
)
if len(primary_thresholded_long) != expected_rows:
    raise ValueError(
        f"Unexpected thresholded rows: {len(primary_thresholded_long)} "
        f"instead of {expected_rows}."
    )

def subgroup_point_metrics(thresholded_predictions, feature_specification):
    enriched = thresholded_predictions.merge(
        data[[
            "id", "sex", "race_ethnicity",
            "income_poverty_group", "insurance_history",
        ]],
        on="id",
        how="left",
        validate="many_to_one",
    )
    absolute_rows, gap_rows = [], []
    for key, group in enriched.groupby(
        ["model", "target", "target_training_sensitivity"],
        observed=True,
        sort=False,
    ):
        model_name, target, sensitivity = key
        for subgroup_variable, levels in SUBGROUP_LEVELS.items():
            reference = REFERENCE_CATEGORIES[subgroup_variable]
            metrics_by_level = {}
            for level in levels:
                level_data = group.loc[group[subgroup_variable].astype(str) == level]
                metrics = threshold_metrics(
                    level_data["observed_outcome"].to_numpy(int),
                    level_data["predicted_class"].to_numpy(int),
                )
                metrics_by_level[level] = metrics
                for metric in THRESHOLD_METRICS:
                    absolute_rows.append({
                        "feature_specification": feature_specification,
                        "model": model_name,
                        "target": target,
                        "target_training_sensitivity": float(sensitivity),
                        "subgroup_variable": subgroup_variable,
                        "subgroup": level,
                        "reference_group": reference,
                        "metric": metric,
                        "estimate": metrics[metric],
                        "subgroup_n": len(level_data),
                        "positive_n": metrics["positive_n"],
                        "negative_n": metrics["negative_n"],
                    })
            reference_metrics = metrics_by_level[reference]
            for level, metrics in metrics_by_level.items():
                for metric in THRESHOLD_METRICS:
                    gap_rows.append({
                        "feature_specification": feature_specification,
                        "model": model_name,
                        "target": target,
                        "target_training_sensitivity": float(sensitivity),
                        "subgroup_variable": subgroup_variable,
                        "subgroup": level,
                        "reference_group": reference,
                        "metric": metric,
                        "gap_from_reference": metrics[metric] - reference_metrics[metric],
                    })
    return pd.DataFrame(absolute_rows), pd.DataFrame(gap_rows)

primary_threshold_absolute, primary_threshold_gaps = subgroup_point_metrics(
    primary_thresholded_long,
    "with_race",
)

keys = [
    "model", "target", "subgroup_variable",
    "subgroup", "reference_group", "metric",
]
threshold_stability = (
    primary_threshold_gaps
    .groupby(keys, as_index=False, observed=True)
    .agg(
        minimum_gap=("gap_from_reference", "min"),
        maximum_gap=("gap_from_reference", "max"),
        gap_range=("gap_from_reference", lambda x: x.max() - x.min()),
    )
)
primary_values = (
    primary_threshold_gaps.loc[
        np.isclose(
            primary_threshold_gaps["target_training_sensitivity"],
            PRIMARY_TARGET_SENSITIVITY,
        ),
        keys + ["gap_from_reference"],
    ]
    .rename(columns={"gap_from_reference": "primary_80_gap"})
)
threshold_stability = threshold_stability.merge(
    primary_values,
    on=keys,
    how="left",
    validate="one_to_one",
)
threshold_stability["gap_sign_changes"] = (
    (threshold_stability["minimum_gap"] < 0)
    & (threshold_stability["maximum_gap"] > 0)
)
range_pp = 100 * threshold_stability["gap_range"]
threshold_stability["threshold_stability_label"] = np.select(
    [
        threshold_stability["gap_sign_changes"],
        range_pp > 10,
        range_pp > 5,
    ],
    [
        "Threshold-dependent: gap changes sign",
        "Threshold-dependent: range above 10 percentage points",
        "Moderately threshold-sensitive",
    ],
    default="Stable across prespecified thresholds",
)

primary_threshold_absolute.to_csv(
    TABLE_DIR / "threshold_sensitivity_absolute_subgroup_metrics.csv",
    index=False,
)
primary_threshold_gaps.to_csv(
    TABLE_DIR / "threshold_sensitivity_reference_gaps.csv",
    index=False,
)
threshold_stability.to_csv(
    TABLE_DIR / "threshold_sensitivity_stability_summary.csv",
    index=False,
)

threshold_stability.sort_values(
    ["gap_sign_changes", "gap_range"],
    ascending=False,
).head(30)


### Threshold-sensitivity appendix figures

A separate figure is created for each subgroup variable. Reference groups are omitted because their gaps are zero by definition.

In [ ]:
threshold_figure_paths = []

for subgroup_variable in SUBGROUP_LEVELS:
    plot_data = primary_threshold_gaps.loc[
        (primary_threshold_gaps["metric"] == "false_negative_rate")
        & (primary_threshold_gaps["subgroup_variable"] == subgroup_variable)
        & (
            primary_threshold_gaps["subgroup"]
            != primary_threshold_gaps["reference_group"]
        )
    ].copy()

    figure, axis = plt.subplots(figsize=(10, 6))
    for key, group in plot_data.groupby(
        ["model", "target", "subgroup"],
        observed=True,
        sort=False,
    ):
        model_name, target, subgroup = key
        group = group.sort_values("target_training_sensitivity")
        axis.plot(
            100 * group["target_training_sensitivity"],
            100 * group["gap_from_reference"],
            marker="o",
            label=(
                f"{MODEL_DISPLAY_NAMES[model_name]} | "
                f"{TARGET_DISPLAY_NAMES[target]} | {subgroup}"
            ),
        )

    axis.axhline(0, linewidth=1, linestyle="--")
    axis.set_xlabel("Training-sensitivity operating point (%)")
    axis.set_ylabel("FNR gap from reference (percentage points)")
    axis.set_title(
        "Threshold sensitivity of FNR gaps: "
        f"{subgroup_variable.replace('_', ' ').title()}"
    )
    axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    figure.tight_layout()

    png = FIGURE_DIR / f"appendix_threshold_sensitivity_fnr_gaps_{subgroup_variable}.png"
    pdf = FIGURE_DIR / f"appendix_threshold_sensitivity_fnr_gaps_{subgroup_variable}.pdf"
    figure.savefig(png, dpi=300, bbox_inches="tight")
    figure.savefig(pdf, bbox_inches="tight")
    plt.close(figure)
    threshold_figure_paths.extend([png, pdf])

print("Saved threshold-sensitivity figures:", len(threshold_figure_paths))


## Section 12A.2 — Complete-case and sample-construction sensitivity

The final corrected selection summary is reconstructed directly from the adult analysis base and the current 4,874-person analytic sample. Exclusion reasons are non-exclusive and therefore do not sum to the total excluded count.

Included-versus-excluded comparisons describe sample composition; they do not identify the direction or magnitude of selection bias.

In [ ]:
analysis_base = analysis_base.copy()
analysis_base["included_in_primary_analysis"] = analysis_base["id"].isin(data["id"])

adult_n = len(analysis_base)
analytic_n = len(data)
excluded_n = adult_n - analytic_n

available_predictors = [
    column for column in PRIMARY_PREDICTORS
    if column in analysis_base.columns
]
available_targets = [
    column for column in TARGET_COLUMNS
    if column in analysis_base.columns
]
survey_fields = [
    column for column in [
        "exam_status", "phlebotomy_weight", "survey_stratum", "survey_psu"
    ]
    if column in analysis_base.columns
]

predictor_missing = analysis_base[available_predictors].isna().any(axis=1)
target_missing = analysis_base[available_targets].isna().any(axis=1)
survey_invalid = analysis_base[survey_fields].isna().any(axis=1)

if "exam_status" in analysis_base.columns:
    survey_invalid |= analysis_base["exam_status"] != 2
if "phlebotomy_weight" in analysis_base.columns:
    survey_invalid |= (
        pd.to_numeric(analysis_base["phlebotomy_weight"], errors="coerce")
        .fillna(0)
        <= 0
    )

pregnancy_exclusion = pd.Series(False, index=analysis_base.index)
if "confirmed_current_pregnancy" in analysis_base.columns:
    pregnancy_exclusion = (
        pd.to_numeric(
            analysis_base["confirmed_current_pregnancy"],
            errors="coerce",
        )
        == 1
    )

selection_summary = pd.DataFrame([
    {
        "stage": "Adult analysis base",
        "n": adult_n,
        "share_of_adult_base": 1.0,
    },
    {
        "stage": "Excluded from primary complete-case analysis",
        "n": excluded_n,
        "share_of_adult_base": excluded_n / adult_n,
    },
    {
        "stage": "Primary analytic sample",
        "n": analytic_n,
        "share_of_adult_base": analytic_n / adult_n,
    },
])

reason_masks = {
    "Confirmed current pregnancy": pregnancy_exclusion,
    "Missing at least one predictor": predictor_missing,
    "Missing at least one target": target_missing,
    "Missing or invalid required examination/survey field": survey_invalid,
}
for column in available_predictors:
    reason_masks[f"Missing predictor: {column}"] = analysis_base[column].isna()
for column in available_targets:
    reason_masks[f"Missing target: {column}"] = analysis_base[column].isna()
for column in survey_fields:
    reason_masks[f"Missing survey field: {column}"] = analysis_base[column].isna()

reason_rows = []
for reason, mask in reason_masks.items():
    mask = mask.fillna(False)
    excluded_with_reason = mask & ~analysis_base["included_in_primary_analysis"]
    reason_rows.append({
        "reason": reason,
        "adult_base_n_with_reason": int(mask.sum()),
        "excluded_n_with_reason": int(excluded_with_reason.sum()),
        "share_of_adult_base": float(mask.mean()),
        "share_of_excluded_adults": (
            float(excluded_with_reason.sum() / excluded_n)
            if excluded_n > 0 else np.nan
        ),
        "nonexclusive_reason": True,
    })
exclusion_reasons = pd.DataFrame(reason_rows)

continuous_rows = []
for variable in CONTINUOUS_PREDICTORS:
    for included, label in [(True, "Included"), (False, "Excluded")]:
        values = pd.to_numeric(
            analysis_base.loc[
                analysis_base["included_in_primary_analysis"] == included,
                variable,
            ],
            errors="coerce",
        )
        observed = values.dropna()
        continuous_rows.append({
            "variable": variable,
            "selection_status": label,
            "group_n": len(values),
            "observed_n": len(observed),
            "missing_n": int(values.isna().sum()),
            "mean": float(observed.mean()) if len(observed) else np.nan,
            "standard_deviation": (
                float(observed.std(ddof=1)) if len(observed) > 1 else np.nan
            ),
            "median": float(observed.median()) if len(observed) else np.nan,
            "q1": float(observed.quantile(0.25)) if len(observed) else np.nan,
            "q3": float(observed.quantile(0.75)) if len(observed) else np.nan,
        })
continuous_selection = pd.DataFrame(continuous_rows)

categorical_rows = []
for variable, levels in {
    "sex": SEX_LEVELS,
    "race_ethnicity": RACE_ETHNICITY_LEVELS,
    "insurance_history": INSURANCE_HISTORY_LEVELS,
}.items():
    for included, label in [(True, "Included"), (False, "Excluded")]:
        values = analysis_base.loc[
            analysis_base["included_in_primary_analysis"] == included,
            variable,
        ]
        observed_n = int(values.notna().sum())
        for level in levels:
            level_n = int((values.astype(str) == level).sum())
            categorical_rows.append({
                "variable": variable,
                "level": level,
                "selection_status": label,
                "group_n": len(values),
                "observed_n": observed_n,
                "level_n": level_n,
                "percent_of_observed": (
                    100 * level_n / observed_n if observed_n else np.nan
                ),
                "missing_n": int(values.isna().sum()),
            })
categorical_selection = pd.DataFrame(categorical_rows)

selection_summary.to_csv(
    TABLE_DIR / "sensitivity_complete_case_selection_summary.csv",
    index=False,
)
exclusion_reasons.to_csv(
    TABLE_DIR / "sensitivity_complete_case_exclusion_reasons.csv",
    index=False,
)
continuous_selection.to_csv(
    TABLE_DIR / "sensitivity_included_excluded_continuous.csv",
    index=False,
)
categorical_selection.to_csv(
    TABLE_DIR / "sensitivity_included_excluded_categorical.csv",
    index=False,
)

display(selection_summary)
display(exclusion_reasons.sort_values("excluded_n_with_reason", ascending=False).head(15))


### Complete-case interpretation

The complete-case analysis is conditional on observed predictors, both targets, and required examination/survey fields. Observed BMI or income among excluded adults does not represent excluded adults whose corresponding value is missing.

Predictor imputation remains optional because a defensible implementation would require fold-specific imputation learned only from training predictors and no use of held-out outcomes.

## Section 12A.3 — Survey-weight sensitivity

Phlebotomy weights are used for descriptive complete-case point estimates only. The notebook does not calculate design-based standard errors from strata and PSUs.

In [ ]:
weights = data["phlebotomy_weight"]
if weights.isna().any() or (weights <= 0).any():
    raise ValueError("Every analytic participant must have a positive phlebotomy weight.")

weighted_rows = []
for variable in ["age", "bmi"]:
    weighted_rows.extend([
        {
            "quantity": f"{variable}: mean",
            "scale": "original units",
            "unweighted_estimate": float(data[variable].mean()),
            "phlebotomy_weighted_complete_case_estimate": weighted_mean(
                data[variable], weights
            ),
        },
        {
            "quantity": f"{variable}: standard deviation",
            "scale": "original units",
            "unweighted_estimate": float(data[variable].std(ddof=1)),
            "phlebotomy_weighted_complete_case_estimate": weighted_sd(
                data[variable], weights
            ),
        },
    ])

for quantile, label in [(0.25, "Q1"), (0.50, "median"), (0.75, "Q3")]:
    weighted_rows.append({
        "quantity": f"income_poverty_ratio: {label}",
        "scale": "original units",
        "unweighted_estimate": float(
            data["income_poverty_ratio"].quantile(quantile)
        ),
        "phlebotomy_weighted_complete_case_estimate": weighted_quantile(
            data["income_poverty_ratio"], weights, quantile
        ),
    })

for target in TARGET_COLUMNS:
    weighted_rows.append({
        "quantity": f"{target}: prevalence",
        "scale": "proportion",
        "unweighted_estimate": float(data[target].mean()),
        "phlebotomy_weighted_complete_case_estimate": weighted_mean(
            data[target], weights
        ),
    })

for joint_label in ["D0_H0", "D0_H1", "D1_H0", "D1_H1"]:
    indicator = (data["joint_label_code"].astype(str) == joint_label).astype(int)
    weighted_rows.append({
        "quantity": f"joint_label_share: {joint_label}",
        "scale": "proportion",
        "unweighted_estimate": float(indicator.mean()),
        "phlebotomy_weighted_complete_case_estimate": weighted_mean(
            indicator, weights
        ),
    })

weighted_unweighted_sensitivity = pd.DataFrame(weighted_rows)
weighted_unweighted_sensitivity["weighted_minus_unweighted"] = (
    weighted_unweighted_sensitivity["phlebotomy_weighted_complete_case_estimate"]
    - weighted_unweighted_sensitivity["unweighted_estimate"]
)
weighted_unweighted_sensitivity["weighting_scope"] = (
    "Phlebotomy-weighted complete-case point estimate; "
    "not a fully design-corrected population estimate"
)
weighted_unweighted_sensitivity.to_csv(
    TABLE_DIR / "sensitivity_weighted_unweighted_descriptive_comparison.csv",
    index=False,
)

weighted_unweighted_sensitivity


## Section 12A.4 — Model-class sensitivity

The central patterns are compared under linear logistic regression and a nonlinear but strictly additive EBM. Differences are interpreted as sensitivity to functional form, not as causal evidence.

In [ ]:
PRIMARY_LOGISTIC_MODEL_PATHS = {
    target: resolve_project_file(f"outputs/models/logistic_{target}.joblib")
    for target in TARGET_COLUMNS
}
PRIMARY_EBM_MODEL_PATHS = {
    target: resolve_project_file(f"outputs/models/ebm_{target}.joblib")
    for target in TARGET_COLUMNS
}
primary_logistic_models = {
    target: joblib.load(path)
    for target, path in PRIMARY_LOGISTIC_MODEL_PATHS.items()
}
primary_ebm_models = {
    target: joblib.load(path)
    for target, path in PRIMARY_EBM_MODEL_PATHS.items()
}

performance_rows = []
for model_name in MODEL_NAMES:
    for target in TARGET_COLUMNS:
        metrics = probability_metrics(
            primary_probabilities[target].to_numpy(int),
            primary_probabilities[
                f"{model_name}__{target}__probability"
            ].to_numpy(float),
        )
        for metric, estimate in metrics.items():
            performance_rows.append({
                "model": model_name,
                "target": target,
                "metric": metric,
                "estimate": estimate,
            })

primary_performance = pd.DataFrame(performance_rows)
model_class_performance = (
    primary_performance
    .pivot(index=["target", "metric"], columns="model", values="estimate")
    .reset_index()
)
model_class_performance["ebm_minus_logistic"] = (
    model_class_performance["ebm"] - model_class_performance["logistic"]
)
model_class_performance.to_csv(
    TABLE_DIR / "sensitivity_model_class_performance.csv",
    index=False,
)

FIXED_CONTRASTS = [
    {
        "contrast_id": "age_40_to_60",
        "predictor": "age",
        "reference": 40.0,
        "comparison": 60.0,
        "display_name": "Age 40 to 60 years",
    },
    {
        "contrast_id": "bmi_25_to_30",
        "predictor": "bmi",
        "reference": 25.0,
        "comparison": 30.0,
        "display_name": "BMI 25 to 30",
    },
    {
        "contrast_id": "income_1_to_2",
        "predictor": "income_poverty_ratio",
        "reference": 1.0,
        "comparison": 2.0,
        "display_name": "Income-to-poverty ratio 1 to 2",
    },
    {
        "contrast_id": "insurance_continuous_to_uninsured",
        "predictor": "insurance_history",
        "reference": "Continuously insured",
        "comparison": "Currently uninsured",
        "display_name": "Continuously insured to currently uninsured",
    },
]

def predict_probability(fitted_model, evaluation_data, predictor_columns):
    return np.asarray(
        fitted_model.predict_proba(evaluation_data[predictor_columns])[:, 1],
        dtype=float,
    )

def average_fixed_contrast(
    fitted_model, evaluation_data, predictor_columns, contrast
):
    reference_data = evaluation_data[predictor_columns].copy()
    comparison_data = evaluation_data[predictor_columns].copy()
    predictor = contrast["predictor"]
    reference_data[predictor] = contrast["reference"]
    comparison_data[predictor] = contrast["comparison"]
    individual = (
        predict_probability(fitted_model, comparison_data, predictor_columns)
        - predict_probability(fitted_model, reference_data, predictor_columns)
    )
    return float(individual.mean()), individual

contrast_rows = []
for target in TARGET_COLUMNS:
    for model_name, fitted_model in [
        ("logistic", primary_logistic_models[target]),
        ("ebm", primary_ebm_models[target]),
    ]:
        for contrast in FIXED_CONTRASTS:
            estimate, _ = average_fixed_contrast(
                fitted_model, data, PRIMARY_PREDICTORS, contrast
            )
            contrast_rows.append({
                "model": model_name,
                "target": target,
                "contrast_id": contrast["contrast_id"],
                "contrast_display_name": contrast["display_name"],
                "estimate_probability": estimate,
                "estimate_percentage_points": 100 * estimate,
            })

model_class_contrasts = pd.DataFrame(contrast_rows)
model_class_contrast_comparison = (
    model_class_contrasts
    .pivot(
        index=["target", "contrast_id", "contrast_display_name"],
        columns="model",
        values="estimate_percentage_points",
    )
    .reset_index()
)
model_class_contrast_comparison["ebm_minus_logistic_percentage_points"] = (
    model_class_contrast_comparison["ebm"]
    - model_class_contrast_comparison["logistic"]
)
model_class_contrast_comparison["same_direction"] = (
    np.sign(model_class_contrast_comparison["ebm"])
    == np.sign(model_class_contrast_comparison["logistic"])
)

model_class_contrasts.to_csv(
    TABLE_DIR / "sensitivity_model_class_fixed_contrasts.csv",
    index=False,
)
model_class_contrast_comparison.to_csv(
    TABLE_DIR / "sensitivity_model_class_fixed_contrast_comparison.csv",
    index=False,
)

primary_80_gaps = primary_threshold_gaps.loc[
    np.isclose(
        primary_threshold_gaps["target_training_sensitivity"],
        PRIMARY_TARGET_SENSITIVITY,
    )
].copy()
model_class_gap_comparison = (
    primary_80_gaps
    .pivot(
        index=[
            "target", "subgroup_variable", "subgroup",
            "reference_group", "metric",
        ],
        columns="model",
        values="gap_from_reference",
    )
    .reset_index()
)
model_class_gap_comparison["ebm_minus_logistic_gap"] = (
    model_class_gap_comparison["ebm"]
    - model_class_gap_comparison["logistic"]
)
model_class_gap_comparison["same_gap_direction"] = (
    np.sign(model_class_gap_comparison["ebm"])
    == np.sign(model_class_gap_comparison["logistic"])
)
model_class_gap_comparison.to_csv(
    TABLE_DIR / "sensitivity_model_class_primary_subgroup_gap_comparison.csv",
    index=False,
)

display(model_class_performance)
display(model_class_contrast_comparison)


## Section 12A.5 — Models with and without race/ethnicity

Four additional sensitivity models are fitted without race/ethnicity:

- logistic regression / prior diagnosis;
- logistic regression / HbA1c;
- additive EBM / prior diagnosis;
- additive EBM / HbA1c.

Everything else remains fixed: participants, folds, continuous preprocessing, model settings, nested threshold selection, operating points, and paired participant bootstrap comparisons. Race/ethnicity remains available for subgroup evaluation.

## 6. Reconstruct the no-race model specifications from saved primary templates

In [ ]:
def make_one_hot_encoder(categories, drop):
    parameters = {
        "categories": categories,
        "drop": drop,
        "handle_unknown": "error",
        "dtype": float,
    }
    if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
        parameters["sparse_output"] = False
    else:
        parameters["sparse"] = False
    return OneHotEncoder(**parameters)

def build_logistic_pipeline(target, include_race):
    if include_race:
        categorical_predictors = PRIMARY_CATEGORICAL_PREDICTORS
        categories = [
            SEX_LEVELS,
            RACE_ETHNICITY_LEVELS,
            INSURANCE_HISTORY_LEVELS,
        ]
        dropped = [
            REFERENCE_CATEGORIES["sex"],
            REFERENCE_CATEGORIES["race_ethnicity"],
            REFERENCE_CATEGORIES["insurance_history"],
        ]
    else:
        categorical_predictors = NO_RACE_CATEGORICAL_PREDICTORS
        categories = [SEX_LEVELS, INSURANCE_HISTORY_LEVELS]
        dropped = [
            REFERENCE_CATEGORIES["sex"],
            REFERENCE_CATEGORIES["insurance_history"],
        ]

    preprocessor = ColumnTransformer(
        transformers=[
            ("continuous", StandardScaler(), CONTINUOUS_PREDICTORS),
            (
                "categorical",
                make_one_hot_encoder(categories, dropped),
                categorical_predictors,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )
    primary_pipeline = primary_logistic_models[target]
    if not isinstance(primary_pipeline, Pipeline):
        raise TypeError("Saved primary logistic model is not a Pipeline.")
    classifier = clone(primary_pipeline.named_steps["model"])
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", classifier),
    ])

def fit_logistic_model(target, training_data, include_race):
    predictors = PRIMARY_PREDICTORS if include_race else NO_RACE_PREDICTORS
    pipeline = build_logistic_pipeline(target, include_race)
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        pipeline.fit(
            training_data[predictors],
            training_data[target].astype(int),
        )
    return pipeline

def primary_ebm_parameters(target):
    supported = set(
        inspect.signature(ExplainableBoostingClassifier).parameters
    )
    return {
        key: value
        for key, value in primary_ebm_models[target].get_params(deep=False).items()
        if key in supported
    }

def build_ebm(target, include_race):
    parameters = primary_ebm_parameters(target)
    if include_race:
        predictors = PRIMARY_PREDICTORS
        feature_types = [
            "continuous", "continuous", "continuous",
            "nominal", "nominal", "nominal",
        ]
    else:
        predictors = NO_RACE_PREDICTORS
        feature_types = [
            "continuous", "continuous", "continuous",
            "nominal", "nominal",
        ]
    parameters["feature_names"] = predictors
    parameters["feature_types"] = feature_types
    parameters["interactions"] = 0
    parameters["random_state"] = RANDOM_STATE
    return ExplainableBoostingClassifier(**parameters)

def fit_ebm_model(target, training_data, include_race):
    predictors = PRIMARY_PREDICTORS if include_race else NO_RACE_PREDICTORS
    model = build_ebm(target, include_race)
    model.fit(
        training_data[predictors],
        training_data[target].astype(int),
    )
    return model

def fit_sensitivity_model(model_name, target, training_data, include_race):
    if model_name == "logistic":
        return fit_logistic_model(target, training_data, include_race)
    if model_name == "ebm":
        return fit_ebm_model(target, training_data, include_race)
    raise KeyError(f"Unknown model: {model_name}")

print("Primary predictors:", PRIMARY_PREDICTORS)
print("No-race predictors:", NO_RACE_PREDICTORS)


## 7. Fixed-fold no-race out-of-fold probabilities

Every participant receives one no-race probability from a model that did not use that participant for fitting.

In [ ]:
NO_RACE_OOF_PATH = PROCESSED_DIR / "no_race_oof_predictions.csv"
NO_RACE_MODEL_PATHS = {
    model_name: {
        target: MODEL_DIR / f"{model_name}_without_race_{target}.joblib"
        for target in TARGET_COLUMNS
    }
    for model_name in MODEL_NAMES
}

if RUN_RACE_OMISSION_ANALYSIS:
    no_race_oof = data[
        ["id", "cv_fold", "joint_label_code", *TARGET_COLUMNS]
    ].copy()
    fit_rows = []

    for model_name in MODEL_NAMES:
        for target in TARGET_COLUMNS:
            probability_column = f"{model_name}__{target}__probability"
            no_race_oof[probability_column] = np.nan

            for outer_fold in range(1, N_SPLITS + 1):
                train = data.loc[data["cv_fold"] != outer_fold]
                test = data.loc[data["cv_fold"] == outer_fold]
                start = time.perf_counter()

                fitted_model = fit_sensitivity_model(
                    model_name, target, train, include_race=False
                )
                probabilities = predict_probability(
                    fitted_model, test, NO_RACE_PREDICTORS
                )
                no_race_oof.loc[
                    test.index, probability_column
                ] = probabilities

                fit_rows.append({
                    "model": model_name,
                    "target": target,
                    "outer_fold": outer_fold,
                    "training_n": len(train),
                    "held_out_n": len(test),
                    "training_positive_n": int(train[target].sum()),
                    "held_out_positive_n": int(test[target].sum()),
                    "elapsed_seconds": time.perf_counter() - start,
                })
                print(
                    f"Completed no-race {model_name}, "
                    f"target={target}, fold={outer_fold}."
                )

    no_race_probability_columns = [
        f"{model}__{target}__probability"
        for model in MODEL_NAMES for target in TARGET_COLUMNS
    ]
    if no_race_oof[no_race_probability_columns].isna().any().any():
        raise ValueError("At least one no-race OOF probability is missing.")
    if not (
        (no_race_oof[no_race_probability_columns] >= 0)
        & (no_race_oof[no_race_probability_columns] <= 1)
    ).all().all():
        raise ValueError("A no-race OOF probability lies outside [0, 1].")

    no_race_oof.to_csv(NO_RACE_OOF_PATH, index=False)
    no_race_fold_fit_summary = pd.DataFrame(fit_rows)
    no_race_fold_fit_summary.to_csv(
        TABLE_DIR / "no_race_fold_fit_summary.csv",
        index=False,
    )
else:
    no_race_oof = pd.read_csv(NO_RACE_OOF_PATH)

print("No-race OOF predictions:", no_race_oof.shape)


## 8. Nested threshold selection for no-race models

For each outer fold, three inner folds are created within the outer training sample, stratified by the four-category joint label. The highest threshold reaching the requested inner-training sensitivity is applied to the original held-out fold probability. Held-out outcomes do not influence their own thresholds.

In [ ]:
NO_RACE_THRESHOLDED_LONG_PATH = (
    PROCESSED_DIR / "no_race_thresholded_oof_predictions_long.csv"
)

if RUN_RACE_OMISSION_ANALYSIS:
    threshold_rows = []
    selection_rows = []
    no_race_oof_by_id = no_race_oof.set_index("id", drop=False)

    for outer_fold in range(1, N_SPLITS + 1):
        outer_train = data.loc[data["cv_fold"] != outer_fold].copy()
        outer_test = data.loc[data["cv_fold"] == outer_fold].copy()

        splitter = StratifiedKFold(
            n_splits=INNER_THRESHOLD_SPLITS,
            shuffle=True,
            random_state=RANDOM_STATE + outer_fold,
        )
        inner_splits = list(
            splitter.split(outer_train, outer_train["joint_label_code"])
        )

        for model_name in MODEL_NAMES:
            for target in TARGET_COLUMNS:
                inner_probability = np.full(len(outer_train), np.nan)

                for inner_train_pos, inner_valid_pos in inner_splits:
                    inner_train = outer_train.iloc[inner_train_pos]
                    inner_valid = outer_train.iloc[inner_valid_pos]
                    inner_model = fit_sensitivity_model(
                        model_name,
                        target,
                        inner_train,
                        include_race=False,
                    )
                    inner_probability[inner_valid_pos] = predict_probability(
                        inner_model,
                        inner_valid,
                        NO_RACE_PREDICTORS,
                    )

                if np.isnan(inner_probability).any():
                    raise ValueError("Inner threshold-training probabilities are missing.")

                probability_column = f"{model_name}__{target}__probability"
                outer_probability = (
                    no_race_oof_by_id.loc[
                        outer_test["id"], probability_column
                    ].to_numpy(float)
                )
                outer_outcome = outer_test[target].to_numpy(int)

                for target_sensitivity in TARGET_SENSITIVITY_LEVELS:
                    threshold = highest_threshold_reaching_sensitivity(
                        outer_train[target].to_numpy(int),
                        inner_probability,
                        target_sensitivity,
                    )
                    inner_class = (inner_probability >= threshold).astype(int)
                    outer_class = (outer_probability >= threshold).astype(int)
                    inner_metrics = threshold_metrics(
                        outer_train[target].to_numpy(int), inner_class
                    )
                    held_out_metrics = threshold_metrics(
                        outer_outcome, outer_class
                    )

                    selection_rows.append({
                        "model": model_name,
                        "target": target,
                        "outer_fold": outer_fold,
                        "target_training_sensitivity": target_sensitivity,
                        "selected_threshold": threshold,
                        "inner_training_sensitivity": inner_metrics["sensitivity"],
                        "inner_training_specificity": inner_metrics["specificity"],
                        "held_out_sensitivity": held_out_metrics["sensitivity"],
                        "held_out_specificity": held_out_metrics["specificity"],
                    })

                    for position, participant_id in enumerate(
                        outer_test["id"].to_numpy()
                    ):
                        threshold_rows.append({
                            "id": participant_id,
                            "cv_fold": outer_fold,
                            "joint_label_code": outer_test.iloc[position][
                                "joint_label_code"
                            ],
                            "model": model_name,
                            "target": target,
                            "target_training_sensitivity": target_sensitivity,
                            "probability": outer_probability[position],
                            "selected_threshold": threshold,
                            "predicted_class": outer_class[position],
                            "observed_outcome": outer_outcome[position],
                            "model_display_name": MODEL_DISPLAY_NAMES[model_name],
                            "target_display_name": TARGET_DISPLAY_NAMES[target],
                        })

    no_race_thresholded_long = pd.DataFrame(threshold_rows)
    no_race_threshold_selection = pd.DataFrame(selection_rows)

    expected_no_race_rows = (
        len(data) * len(MODEL_NAMES) * len(TARGET_COLUMNS)
        * len(TARGET_SENSITIVITY_LEVELS)
    )
    if len(no_race_thresholded_long) != expected_no_race_rows:
        raise ValueError("Unexpected number of no-race thresholded rows.")

    no_race_thresholded_long.to_csv(
        NO_RACE_THRESHOLDED_LONG_PATH,
        index=False,
    )
    no_race_threshold_selection.to_csv(
        TABLE_DIR / "no_race_fold_specific_threshold_selection.csv",
        index=False,
    )
else:
    no_race_thresholded_long = pd.read_csv(
        NO_RACE_THRESHOLDED_LONG_PATH
    )

print("No-race thresholded predictions:", no_race_thresholded_long.shape)


## 9. With-race versus without-race probability performance

Performance is calculated from out-of-fold probabilities. Paired bootstrap resampling uses the same participant indices for the with-race and without-race versions.

The reported difference is

$$
\text{without-race metric}
-
\text{with-race metric}.
$$

In [ ]:
performance_rows = []
for model_name in MODEL_NAMES:
    for target in TARGET_COLUMNS:
        outcome = data[target].to_numpy(int)
        for specification, probability in [
            (
                "with_race",
                primary_probabilities[
                    f"{model_name}__{target}__probability"
                ].to_numpy(float),
            ),
            (
                "without_race",
                no_race_oof[
                    f"{model_name}__{target}__probability"
                ].to_numpy(float),
            ),
        ]:
            metrics = probability_metrics(outcome, probability)
            for metric, estimate in metrics.items():
                performance_rows.append({
                    "feature_specification": specification,
                    "model": model_name,
                    "target": target,
                    "metric": metric,
                    "estimate": estimate,
                })

race_spec_probability_performance = pd.DataFrame(performance_rows)
race_spec_probability_wide = (
    race_spec_probability_performance
    .pivot(
        index=["model", "target", "metric"],
        columns="feature_specification",
        values="estimate",
    )
    .reset_index()
)
race_spec_probability_wide["without_minus_with_race"] = (
    race_spec_probability_wide["without_race"]
    - race_spec_probability_wide["with_race"]
)

race_spec_probability_performance.to_csv(
    TABLE_DIR / "race_omission_probability_performance_long.csv",
    index=False,
)
race_spec_probability_wide.to_csv(
    TABLE_DIR / "race_omission_probability_performance_comparison.csv",
    index=False,
)

race_spec_probability_wide


In [ ]:
rng_performance = np.random.default_rng(RANDOM_STATE + 8000)
bootstrap_rows = []

for model_name in MODEL_NAMES:
    for target in TARGET_COLUMNS:
        outcome = data[target].to_numpy(int)
        with_probability = primary_probabilities[
            f"{model_name}__{target}__probability"
        ].to_numpy(float)
        without_probability = no_race_oof[
            f"{model_name}__{target}__probability"
        ].to_numpy(float)

        metric_names = list(probability_metrics(outcome, with_probability))
        with_values = {
            metric: np.full(N_BOOTSTRAP, np.nan) for metric in metric_names
        }
        without_values = {
            metric: np.full(N_BOOTSTRAP, np.nan) for metric in metric_names
        }
        difference_values = {
            metric: np.full(N_BOOTSTRAP, np.nan) for metric in metric_names
        }

        for bootstrap_index in range(N_BOOTSTRAP):
            sampled = rng_performance.integers(
                0, len(data), size=len(data)
            )
            sampled_outcome = outcome[sampled]
            if np.unique(sampled_outcome).size < 2:
                continue

            with_metrics = probability_metrics(
                sampled_outcome, with_probability[sampled]
            )
            without_metrics = probability_metrics(
                sampled_outcome, without_probability[sampled]
            )
            for metric in metric_names:
                with_values[metric][bootstrap_index] = with_metrics[metric]
                without_values[metric][bootstrap_index] = without_metrics[metric]
                difference_values[metric][bootstrap_index] = (
                    without_metrics[metric] - with_metrics[metric]
                )

        for metric in metric_names:
            point = race_spec_probability_wide.loc[
                (race_spec_probability_wide["model"] == model_name)
                & (race_spec_probability_wide["target"] == target)
                & (race_spec_probability_wide["metric"] == metric)
            ].iloc[0]
            with_lower, with_upper = percentile_interval(with_values[metric])
            without_lower, without_upper = percentile_interval(
                without_values[metric]
            )
            difference_lower, difference_upper = percentile_interval(
                difference_values[metric]
            )
            bootstrap_rows.append({
                "model": model_name,
                "target": target,
                "metric": metric,
                "with_race_estimate": point["with_race"],
                "with_race_ci_lower": with_lower,
                "with_race_ci_upper": with_upper,
                "without_race_estimate": point["without_race"],
                "without_race_ci_lower": without_lower,
                "without_race_ci_upper": without_upper,
                "without_minus_with_estimate": point[
                    "without_minus_with_race"
                ],
                "without_minus_with_ci_lower": difference_lower,
                "without_minus_with_ci_upper": difference_upper,
                "bootstrap_replicates_requested": N_BOOTSTRAP,
                "bootstrap_replicates_valid": int(
                    np.isfinite(difference_values[metric]).sum()
                ),
            })

race_omission_performance_with_intervals = pd.DataFrame(bootstrap_rows)
race_omission_performance_with_intervals.to_csv(
    TABLE_DIR / "race_omission_probability_performance_with_intervals.csv",
    index=False,
)

race_omission_performance_with_intervals


## 10. Full-sample no-race models and probability-scale effects

Full-sample models are used only for interpretation. Primary predictive performance remains based on out-of-fold probabilities.

For continuous predictors:

- logistic regression uses the analytical one-SD average marginal effect definition from Notebook 04;
- EBM uses a centred one-SD finite probability contrast because its nonlinear additive shape has no single global linear derivative.

For categorical predictors, both model classes use average discrete probability changes.

The effect bootstrap conditions on the fitted models and resamples participant-level effects. It does not represent full model-refit uncertainty.

In [ ]:
full_no_race_models = {}

if RUN_RACE_OMISSION_ANALYSIS:
    for model_name in MODEL_NAMES:
        full_no_race_models[model_name] = {}
        for target in TARGET_COLUMNS:
            fitted_model = fit_sensitivity_model(
                model_name, target, data, include_race=False
            )
            full_no_race_models[model_name][target] = fitted_model
            joblib.dump(
                fitted_model,
                NO_RACE_MODEL_PATHS[model_name][target],
            )
            print(f"Saved full no-race {model_name} model for {target}.")
else:
    full_no_race_models = {
        model_name: {
            target: joblib.load(NO_RACE_MODEL_PATHS[model_name][target])
            for target in TARGET_COLUMNS
        }
        for model_name in MODEL_NAMES
    }

EFFECT_SPECIFICATIONS = [
    {
        "effect_id": "age_1sd",
        "predictor": "age",
        "effect_family": "continuous",
        "display_name": "Age: one-SD effect",
    },
    {
        "effect_id": "bmi_1sd",
        "predictor": "bmi",
        "effect_family": "continuous",
        "display_name": "BMI: one-SD effect",
    },
    {
        "effect_id": "income_1sd",
        "predictor": "income_poverty_ratio",
        "effect_family": "continuous",
        "display_name": "Income-to-poverty ratio: one-SD effect",
    },
    {
        "effect_id": "sex_male_vs_female",
        "predictor": "sex",
        "effect_family": "categorical",
        "reference": "Female",
        "comparison": "Male",
        "display_name": "Male versus female",
    },
    {
        "effect_id": "insurance_gap_vs_continuous",
        "predictor": "insurance_history",
        "effect_family": "categorical",
        "reference": "Continuously insured",
        "comparison": "Currently insured, past-year gap",
        "display_name": "Past-year insurance gap versus continuously insured",
    },
    {
        "effect_id": "uninsured_vs_continuous",
        "predictor": "insurance_history",
        "effect_family": "categorical",
        "reference": "Continuously insured",
        "comparison": "Currently uninsured",
        "display_name": "Currently uninsured versus continuously insured",
    },
]

def logistic_individual_effect(
    fitted_pipeline, evaluation_data, predictors, specification
):
    predictor = specification["predictor"]
    if specification["effect_family"] == "categorical":
        reference_data = evaluation_data[predictors].copy()
        comparison_data = evaluation_data[predictors].copy()
        reference_data[predictor] = specification["reference"]
        comparison_data[predictor] = specification["comparison"]
        individual = (
            predict_probability(fitted_pipeline, comparison_data, predictors)
            - predict_probability(fitted_pipeline, reference_data, predictors)
        )
        return individual, "Average discrete probability change"

    probability = predict_probability(
        fitted_pipeline, evaluation_data, predictors
    )
    feature_names = (
        fitted_pipeline.named_steps["preprocessor"].get_feature_names_out()
    )
    coefficient = fitted_pipeline.named_steps["model"].coef_[0]
    positions = [
        position for position, feature_name in enumerate(feature_names)
        if feature_name == f"continuous__{predictor}"
        or feature_name.endswith(f"__{predictor}")
    ]
    if len(positions) != 1:
        raise RuntimeError(
            f"Could not uniquely locate logistic coefficient for {predictor}."
        )
    individual = (
        probability * (1 - probability) * coefficient[positions[0]]
    )
    return individual, "Analytical average marginal effect for one SD"

def ebm_individual_effect(
    fitted_model, evaluation_data, predictors, specification
):
    predictor = specification["predictor"]
    reference_data = evaluation_data[predictors].copy()
    comparison_data = evaluation_data[predictors].copy()

    if specification["effect_family"] == "categorical":
        reference_data[predictor] = specification["reference"]
        comparison_data[predictor] = specification["comparison"]
        definition = "Average discrete probability change"
    else:
        sd = float(evaluation_data[predictor].std(ddof=1))
        lower = float(evaluation_data[predictor].min())
        upper = float(evaluation_data[predictor].max())
        reference_data[predictor] = np.clip(
            evaluation_data[predictor] - 0.5 * sd, lower, upper
        )
        comparison_data[predictor] = np.clip(
            evaluation_data[predictor] + 0.5 * sd, lower, upper
        )
        definition = "Centred average one-SD finite probability contrast"

    individual = (
        predict_probability(fitted_model, comparison_data, predictors)
        - predict_probability(fitted_model, reference_data, predictors)
    )
    return individual, definition

effect_individual_values = {}
effect_rows = []

for model_name in MODEL_NAMES:
    for target in TARGET_COLUMNS:
        model_specs = [
            (
                "with_race",
                primary_logistic_models[target]
                if model_name == "logistic"
                else primary_ebm_models[target],
                PRIMARY_PREDICTORS,
            ),
            (
                "without_race",
                full_no_race_models[model_name][target],
                NO_RACE_PREDICTORS,
            ),
        ]
        for feature_specification, fitted_model, predictors in model_specs:
            for specification in EFFECT_SPECIFICATIONS:
                if model_name == "logistic":
                    individual, definition = logistic_individual_effect(
                        fitted_model, data, predictors, specification
                    )
                else:
                    individual, definition = ebm_individual_effect(
                        fitted_model, data, predictors, specification
                    )
                key = (
                    model_name, target,
                    feature_specification, specification["effect_id"],
                )
                effect_individual_values[key] = np.asarray(individual, dtype=float)
                effect_rows.append({
                    "model": model_name,
                    "target": target,
                    "feature_specification": feature_specification,
                    "effect_id": specification["effect_id"],
                    "effect_display_name": specification["display_name"],
                    "predictor": specification["predictor"],
                    "effect_definition": definition,
                    "estimate_probability": float(np.mean(individual)),
                    "estimate_percentage_points": float(100 * np.mean(individual)),
                })

race_omission_effects = pd.DataFrame(effect_rows)
race_omission_effects.to_csv(
    TABLE_DIR / "race_omission_probability_effects_long.csv",
    index=False,
)
race_omission_effects.head(20)


In [ ]:
rng_effects = np.random.default_rng(RANDOM_STATE + 8100)
effect_comparison_rows = []

for model_name in MODEL_NAMES:
    for target in TARGET_COLUMNS:
        for specification in EFFECT_SPECIFICATIONS:
            effect_id = specification["effect_id"]
            with_values = effect_individual_values[
                (model_name, target, "with_race", effect_id)
            ]
            without_values = effect_individual_values[
                (model_name, target, "without_race", effect_id)
            ]
            bootstrap_with = np.full(N_BOOTSTRAP, np.nan)
            bootstrap_without = np.full(N_BOOTSTRAP, np.nan)
            bootstrap_difference = np.full(N_BOOTSTRAP, np.nan)

            for bootstrap_index in range(N_BOOTSTRAP):
                sampled = rng_effects.integers(
                    0, len(data), size=len(data)
                )
                bootstrap_with[bootstrap_index] = with_values[sampled].mean()
                bootstrap_without[bootstrap_index] = (
                    without_values[sampled].mean()
                )
                bootstrap_difference[bootstrap_index] = (
                    without_values[sampled].mean()
                    - with_values[sampled].mean()
                )

            with_lower, with_upper = percentile_interval(bootstrap_with)
            without_lower, without_upper = percentile_interval(
                bootstrap_without
            )
            difference_lower, difference_upper = percentile_interval(
                bootstrap_difference
            )

            effect_comparison_rows.append({
                "model": model_name,
                "target": target,
                "effect_id": effect_id,
                "effect_display_name": specification["display_name"],
                "predictor": specification["predictor"],
                "with_race_estimate_percentage_points": 100 * with_values.mean(),
                "with_race_ci_lower_percentage_points": 100 * with_lower,
                "with_race_ci_upper_percentage_points": 100 * with_upper,
                "without_race_estimate_percentage_points": (
                    100 * without_values.mean()
                ),
                "without_race_ci_lower_percentage_points": 100 * without_lower,
                "without_race_ci_upper_percentage_points": 100 * without_upper,
                "without_minus_with_percentage_points": (
                    100 * (without_values.mean() - with_values.mean())
                ),
                "difference_ci_lower_percentage_points": 100 * difference_lower,
                "difference_ci_upper_percentage_points": 100 * difference_upper,
                "uncertainty_scope": (
                    "Paired participant bootstrap of individual effects, "
                    "conditional on fitted full-sample models"
                ),
            })

race_omission_effect_comparison = pd.DataFrame(effect_comparison_rows)
race_omission_effect_comparison.to_csv(
    TABLE_DIR / "race_omission_probability_effect_comparison.csv",
    index=False,
)
race_omission_effect_comparison


## 11. With-race versus without-race subgroup metrics

Absolute subgroup FNR, FPR, and positive prediction rates, together with gaps from the prespecified reference groups, are calculated at 70%, 80%, and 90%. Full tables belong in the appendix.

In [ ]:
no_race_threshold_absolute, no_race_threshold_gaps = subgroup_point_metrics(
    no_race_thresholded_long,
    "without_race",
)
race_spec_threshold_absolute = pd.concat(
    [primary_threshold_absolute, no_race_threshold_absolute],
    ignore_index=True,
)
race_spec_threshold_gaps = pd.concat(
    [primary_threshold_gaps, no_race_threshold_gaps],
    ignore_index=True,
)

absolute_index = [
    "model", "target", "target_training_sensitivity",
    "subgroup_variable", "subgroup", "reference_group",
    "metric", "subgroup_n", "positive_n", "negative_n",
]
absolute_comparison = (
    race_spec_threshold_absolute
    .pivot(
        index=absolute_index,
        columns="feature_specification",
        values="estimate",
    )
    .reset_index()
)
absolute_comparison["without_minus_with_race"] = (
    absolute_comparison["without_race"]
    - absolute_comparison["with_race"]
)

gap_index = [
    "model", "target", "target_training_sensitivity",
    "subgroup_variable", "subgroup", "reference_group", "metric",
]
gap_comparison = (
    race_spec_threshold_gaps
    .pivot(
        index=gap_index,
        columns="feature_specification",
        values="gap_from_reference",
    )
    .reset_index()
)
gap_comparison["without_minus_with_race_gap"] = (
    gap_comparison["without_race"]
    - gap_comparison["with_race"]
)

race_spec_threshold_absolute.to_csv(
    TABLE_DIR / "race_omission_subgroup_metrics_long.csv",
    index=False,
)
race_spec_threshold_gaps.to_csv(
    TABLE_DIR / "race_omission_subgroup_gaps_long.csv",
    index=False,
)
absolute_comparison.to_csv(
    TABLE_DIR / "race_omission_absolute_subgroup_metric_comparison.csv",
    index=False,
)
gap_comparison.to_csv(
    TABLE_DIR / "race_omission_subgroup_gap_comparison.csv",
    index=False,
)

gap_comparison.loc[
    np.isclose(
        gap_comparison["target_training_sensitivity"],
        PRIMARY_TARGET_SENSITIVITY,
    )
].sort_values(
    "without_minus_with_race_gap",
    key=lambda x: x.abs(),
    ascending=False,
).head(30)


### Paired bootstrap for subgroup consequences of race omission

The bootstrap compares with-race and without-race predictions for the same resampled participants. Small subgroup cells remain unstable even when a paired difference is reported.

In [ ]:
ordered_data = data.sort_values("id").reset_index(drop=True)
ordered_ids = ordered_data["id"].to_numpy()

def keyed_threshold_predictions(frame):
    return {
        (model, target, float(sensitivity)): group.sort_values("id")
        for (model, target, sensitivity), group in frame.groupby(
            ["model", "target", "target_training_sensitivity"],
            observed=True,
        )
    }

primary_keyed = keyed_threshold_predictions(primary_thresholded_long)
no_race_keyed = keyed_threshold_predictions(no_race_thresholded_long)

for key in primary_keyed:
    if not np.array_equal(primary_keyed[key]["id"].to_numpy(), ordered_ids):
        raise ValueError(f"Primary thresholded IDs are misaligned for {key}.")
    if not np.array_equal(no_race_keyed[key]["id"].to_numpy(), ordered_ids):
        raise ValueError(f"No-race thresholded IDs are misaligned for {key}.")

def metrics_by_code(outcome, predicted, code, level_n):
    y = np.asarray(outcome, dtype=int)
    pred = np.asarray(predicted, dtype=int)
    code = np.asarray(code, dtype=int)
    positive = np.bincount(
        code, weights=(y == 1).astype(float), minlength=level_n
    )
    negative = np.bincount(
        code, weights=(y == 0).astype(float), minlength=level_n
    )
    fn = np.bincount(
        code,
        weights=((y == 1) & (pred == 0)).astype(float),
        minlength=level_n,
    )
    fp = np.bincount(
        code,
        weights=((y == 0) & (pred == 1)).astype(float),
        minlength=level_n,
    )
    predicted_positive = np.bincount(
        code, weights=(pred == 1).astype(float), minlength=level_n
    )
    total = np.bincount(code, minlength=level_n).astype(float)
    return np.column_stack([
        safe_divide(fn, positive),
        safe_divide(fp, negative),
        safe_divide(predicted_positive, total),
    ])

metric_position = {
    "false_negative_rate": 0,
    "false_positive_rate": 1,
    "positive_prediction_rate": 2,
}
rng_fairness = np.random.default_rng(RANDOM_STATE + 8200)
fairness_bootstrap_rows = []

for key in sorted(primary_keyed):
    model_name, target, sensitivity = key
    with_group = primary_keyed[key]
    without_group = no_race_keyed[key]
    outcome = with_group["observed_outcome"].to_numpy(int)
    with_pred = with_group["predicted_class"].to_numpy(int)
    without_pred = without_group["predicted_class"].to_numpy(int)

    if not np.array_equal(
        outcome, without_group["observed_outcome"].to_numpy(int)
    ):
        raise ValueError(f"Outcomes differ across race specifications for {key}.")

    for subgroup_variable, levels in SUBGROUP_LEVELS.items():
        codes = pd.Categorical(
            ordered_data[subgroup_variable].astype(str),
            categories=levels,
            ordered=True,
        ).codes
        if (codes < 0).any():
            raise ValueError(f"Unknown level in {subgroup_variable}.")
        level_n = len(levels)
        reference_position = levels.index(
            REFERENCE_CATEGORIES[subgroup_variable]
        )
        absolute_bootstrap = np.full(
            (N_BOOTSTRAP, level_n, len(THRESHOLD_METRICS)), np.nan
        )
        gap_bootstrap = np.full_like(absolute_bootstrap, np.nan)

        for bootstrap_index in range(N_BOOTSTRAP):
            sampled = rng_fairness.integers(
                0, len(ordered_data), size=len(ordered_data)
            )
            sampled_code = codes[sampled]
            sampled_outcome = outcome[sampled]
            with_metrics = metrics_by_code(
                sampled_outcome, with_pred[sampled], sampled_code, level_n
            )
            without_metrics = metrics_by_code(
                sampled_outcome, without_pred[sampled], sampled_code, level_n
            )
            absolute_bootstrap[bootstrap_index] = (
                without_metrics - with_metrics
            )
            with_gap = with_metrics - with_metrics[reference_position]
            without_gap = (
                without_metrics - without_metrics[reference_position]
            )
            gap_bootstrap[bootstrap_index] = without_gap - with_gap

        point_absolute = absolute_comparison.loc[
            (absolute_comparison["model"] == model_name)
            & (absolute_comparison["target"] == target)
            & np.isclose(
                absolute_comparison["target_training_sensitivity"],
                sensitivity,
            )
            & (
                absolute_comparison["subgroup_variable"]
                == subgroup_variable
            )
        ].set_index(["subgroup", "metric"])

        point_gap = gap_comparison.loc[
            (gap_comparison["model"] == model_name)
            & (gap_comparison["target"] == target)
            & np.isclose(
                gap_comparison["target_training_sensitivity"],
                sensitivity,
            )
            & (gap_comparison["subgroup_variable"] == subgroup_variable)
        ].set_index(["subgroup", "metric"])

        if not point_absolute.index.is_unique:
            raise ValueError(
                "Absolute subgroup point estimates are not unique by "
                "subgroup and metric."
            )

        if not point_gap.index.is_unique:
            raise ValueError(
                "Subgroup-gap point estimates are not unique by "
                "subgroup and metric."
            )

        for level_position, level in enumerate(levels):
            for metric in THRESHOLD_METRICS:
                position = metric_position[metric]
                abs_lower, abs_upper = percentile_interval(
                    absolute_bootstrap[:, level_position, position]
                )
                gap_lower, gap_upper = percentile_interval(
                    gap_bootstrap[:, level_position, position]
                )
                fairness_bootstrap_rows.append({
                    "model": model_name,
                    "target": target,
                    "target_training_sensitivity": sensitivity,
                    "subgroup_variable": subgroup_variable,
                    "subgroup": level,
                    "reference_group": REFERENCE_CATEGORIES[subgroup_variable],
                    "metric": metric,
                    "without_minus_with_absolute_estimate": float(
                        point_absolute.loc[
                            (level, metric),
                            "without_minus_with_race",
                        ]
                    ),
                    "absolute_difference_ci_lower": abs_lower,
                    "absolute_difference_ci_upper": abs_upper,
                    "without_minus_with_gap_estimate": float(
                        point_gap.loc[
                            (level, metric),
                            "without_minus_with_race_gap",
                        ]
                    ),
                    "gap_difference_ci_lower": gap_lower,
                    "gap_difference_ci_upper": gap_upper,
                    "bootstrap_replicates_requested": N_BOOTSTRAP,
                    "absolute_valid_replicates": int(
                        np.isfinite(
                            absolute_bootstrap[:, level_position, position]
                        ).sum()
                    ),
                    "gap_valid_replicates": int(
                        np.isfinite(
                            gap_bootstrap[:, level_position, position]
                        ).sum()
                    ),
                })

race_omission_fairness_with_intervals = pd.DataFrame(
    fairness_bootstrap_rows
)
race_omission_fairness_with_intervals.to_csv(
    TABLE_DIR / "race_omission_subgroup_changes_with_intervals.csv",
    index=False,
)

race_omission_fairness_with_intervals.loc[
    np.isclose(
        race_omission_fairness_with_intervals[
            "target_training_sensitivity"
        ],
        PRIMARY_TARGET_SENSITIVITY,
    )
].sort_values(
    "without_minus_with_gap_estimate",
    key=lambda x: x.abs(),
    ascending=False,
).head(30)


## 12. Threshold stability of race-omission consequences

In [ ]:
race_omission_gap_stability = (
    gap_comparison
    .groupby(
        [
            "model", "target", "subgroup_variable",
            "subgroup", "reference_group", "metric",
        ],
        as_index=False,
        observed=True,
    )
    .agg(
        minimum_change=("without_minus_with_race_gap", "min"),
        maximum_change=("without_minus_with_race_gap", "max"),
        change_range=(
            "without_minus_with_race_gap",
            lambda x: x.max() - x.min(),
        ),
    )
)
race_omission_gap_stability["change_signs_across_thresholds"] = (
    (race_omission_gap_stability["minimum_change"] < 0)
    & (race_omission_gap_stability["maximum_change"] > 0)
)
change_range_pp = 100 * race_omission_gap_stability["change_range"]
race_omission_gap_stability["stability_label"] = np.select(
    [
        race_omission_gap_stability["change_signs_across_thresholds"],
        change_range_pp > 10,
        change_range_pp > 5,
    ],
    [
        "Threshold-dependent: change reverses direction",
        "Threshold-dependent: range above 10 percentage points",
        "Moderately threshold-sensitive",
    ],
    default="Stable across prespecified thresholds",
)
race_omission_gap_stability.to_csv(
    TABLE_DIR / "race_omission_threshold_stability_summary.csv",
    index=False,
)
race_omission_gap_stability.sort_values(
    "change_range", ascending=False
).head(30)


### Appendix figure: race-subgroup FNRs with and without race/ethnicity

The figure is restricted to the primary 80% operating point. It belongs in the appendix because Figure 3 already presents the primary with-race subgroup analysis.

In [ ]:
race_fnr_plot = race_spec_threshold_absolute.loc[
    (race_spec_threshold_absolute["metric"] == "false_negative_rate")
    & (
        race_spec_threshold_absolute["subgroup_variable"]
        == "race_ethnicity"
    )
    & np.isclose(
        race_spec_threshold_absolute["target_training_sensitivity"],
        PRIMARY_TARGET_SENSITIVITY,
    )
].copy()

figure, axis = plt.subplots(figsize=(11, 7))
x_positions = np.arange(len(RACE_ETHNICITY_LEVELS))
series_keys = list(
    race_fnr_plot[
        ["model", "target", "feature_specification"]
    ].drop_duplicates().itertuples(index=False, name=None)
)
offset_values = np.linspace(-0.32, 0.32, len(series_keys))
offsets = dict(zip(series_keys, offset_values))

for key, group in race_fnr_plot.groupby(
    ["model", "target", "feature_specification"],
    observed=True,
    sort=False,
):
    model_name, target, specification = key
    group = group.set_index("subgroup").reindex(RACE_ETHNICITY_LEVELS)
    axis.scatter(
        x_positions + offsets[key],
        100 * group["estimate"],
        label=(
            f"{MODEL_DISPLAY_NAMES[model_name]} | "
            f"{TARGET_DISPLAY_NAMES[target]} | "
            f"{specification.replace('_', ' ')}"
        ),
        s=45,
    )

axis.set_xticks(
    x_positions,
    labels=RACE_ETHNICITY_LEVELS,
    rotation=35,
    ha="right",
)
axis.set_ylabel("False-negative rate (%)")
axis.set_xlabel("Race/ethnicity subgroup")
axis.set_title(
    "Appendix: race-subgroup FNRs with and without race/ethnicity "
    "at the 80% operating point"
)
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
figure.tight_layout()

race_omission_figure_png = (
    FIGURE_DIR / "appendix_race_omission_race_subgroup_fnr.png"
)
race_omission_figure_pdf = (
    FIGURE_DIR / "appendix_race_omission_race_subgroup_fnr.pdf"
)
figure.savefig(race_omission_figure_png, dpi=300, bbox_inches="tight")
figure.savefig(race_omission_figure_pdf, bbox_inches="tight")
plt.close(figure)

print("Saved:", race_omission_figure_png)
print("Saved:", race_omission_figure_pdf)


## Interpretation constraints for race omission

Omitting race/ethnicity from the predictor matrix is a sensitivity analysis, not a fairness guarantee. Race/ethnicity remains available for subgroup auditing, and changes in performance or gaps are interpreted descriptively rather than causally.

## Reproducibility metadata and Notebook 08 checkpoint

Participant-level data, out-of-fold probabilities, and thresholded predictions should remain excluded from public GitHub commits. Notebook 09 loads the saved aggregate sensitivity outputs and creates the publication-ready tables and figures.

In [ ]:
required_notebook08_outputs = [
    TABLE_DIR / "threshold_sensitivity_absolute_subgroup_metrics.csv",
    TABLE_DIR / "threshold_sensitivity_reference_gaps.csv",
    TABLE_DIR / "threshold_sensitivity_stability_summary.csv",
    TABLE_DIR / "sensitivity_complete_case_selection_summary.csv",
    TABLE_DIR / "sensitivity_complete_case_exclusion_reasons.csv",
    TABLE_DIR / "sensitivity_included_excluded_continuous.csv",
    TABLE_DIR / "sensitivity_included_excluded_categorical.csv",
    TABLE_DIR / "sensitivity_weighted_unweighted_descriptive_comparison.csv",
    TABLE_DIR / "sensitivity_model_class_performance.csv",
    TABLE_DIR / "sensitivity_model_class_fixed_contrasts.csv",
    TABLE_DIR / "sensitivity_model_class_fixed_contrast_comparison.csv",
    TABLE_DIR / "sensitivity_model_class_primary_subgroup_gap_comparison.csv",
    TABLE_DIR / "race_omission_probability_performance_with_intervals.csv",
    TABLE_DIR / "race_omission_probability_effect_comparison.csv",
    TABLE_DIR / "race_omission_subgroup_metrics_long.csv",
    TABLE_DIR / "race_omission_subgroup_gaps_long.csv",
    TABLE_DIR / "race_omission_subgroup_changes_with_intervals.csv",
    TABLE_DIR / "race_omission_threshold_stability_summary.csv",
    NO_RACE_OOF_PATH,
    NO_RACE_THRESHOLDED_LONG_PATH,
    race_omission_figure_png,
    race_omission_figure_pdf,
]

missing_outputs = [
    path for path in required_notebook08_outputs if not path.exists()
]
if missing_outputs:
    raise FileNotFoundError(
        "Notebook 08 did not create all required sensitivity outputs:\n"
        + "\n".join(f"- {path}" for path in missing_outputs)
    )

metadata = {
    "analytic_sample_n": int(len(data)),
    "adult_analysis_base_n": int(len(analysis_base)),
    "complete_case_retention_share": float(len(data) / len(analysis_base)),
    "random_state": RANDOM_STATE,
    "outer_folds": N_SPLITS,
    "inner_threshold_folds": INNER_THRESHOLD_SPLITS,
    "target_sensitivity_levels": TARGET_SENSITIVITY_LEVELS,
    "primary_target_sensitivity": PRIMARY_TARGET_SENSITIVITY,
    "bootstrap_replicates": N_BOOTSTRAP,
    "target_columns": TARGET_COLUMNS,
    "primary_predictors": PRIMARY_PREDICTORS,
    "no_race_predictors": NO_RACE_PREDICTORS,
    "same_participants_for_race_omission": True,
    "same_outer_folds_for_race_omission": True,
    "race_retained_for_subgroup_evaluation": True,
    "logistic_template_reused": True,
    "ebm_template_reused": True,
    "ebm_interactions_in_no_race_models": 0,
    "held_out_outcomes_used_for_threshold_selection": False,
    "weighted_descriptive_scope": (
        "Phlebotomy-weighted complete-case point estimates"
    ),
    "upstream_file_hashes": {
        "labelled_data": sha256_file(LABELLED_DATA_PATH),
        "fold_assignments": sha256_file(FOLD_ASSIGNMENT_PATH),
        "logistic_oof": sha256_file(LOGISTIC_OOF_PATH),
        "ebm_oof": sha256_file(EBM_OOF_PATH),
        "thresholded_oof": sha256_file(THRESHOLDED_LONG_PATH),
    },
    "software_versions": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
        "interpret": interpret.__version__,
    },
}

NOTEBOOK08_METADATA_PATH = (
    PROCESSED_DIR / "sensitivity_analysis_metadata.json"
)
with NOTEBOOK08_METADATA_PATH.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

no_race_probability_columns = [
    f"{model}__{target}__probability"
    for model in MODEL_NAMES for target in TARGET_COLUMNS
]
expected_no_race_thresholded_rows = (
    len(data) * len(MODEL_NAMES) * len(TARGET_COLUMNS)
    * len(TARGET_SENSITIVITY_LEVELS)
)

notebook08_checkpoint = {
    "analytic_sample_n": len(data),
    "adult_analysis_base_n": len(analysis_base),
    "same_five_folds_reused": (
        sorted(data["cv_fold"].unique()) == list(range(1, N_SPLITS + 1))
    ),
    "no_race_oof_probabilities_complete": (
        not no_race_oof[no_race_probability_columns].isna().any().any()
    ),
    "no_race_thresholded_rows": len(no_race_thresholded_long),
    "expected_no_race_thresholded_rows": expected_no_race_thresholded_rows,
    "bootstrap_replicates_requested": N_BOOTSTRAP,
    "race_retained_for_subgroup_evaluation": True,
    "all_required_sensitivity_outputs_saved": len(missing_outputs) == 0,
    "metadata_saved": NOTEBOOK08_METADATA_PATH.exists(),
}

print("Saved Notebook 08 metadata to:")
print(NOTEBOOK08_METADATA_PATH)
print()
print("Notebook 08 checkpoint:")
notebook08_checkpoint


## Completion criteria

- Primary samples, folds, target definitions, and sensitivity specifications remain unchanged.
- Race/ethnicity remains available for auditing when omitted from predictors.
- Every required sensitivity output and the Notebook 08 checkpoint are created.